# Day 34: LLM Deployment and Lifecycle
**Hosting choices, versioning and rollback, an AI gateway, stateless MCP with short-lived tokens, streaming and canary releases**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Install

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai tiktoken scikit-learn numpy pandas fastapi httpx uvicorn nest_asyncio

# ===== Option B: VS Code / local =====
# pip install openai tiktoken scikit-learn numpy pandas fastapi httpx uvicorn nest_asyncio
# Create a .env file (git-ignored) or set the variable in your terminal:  OPENAI_API_KEY=...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

In [ ]:
import asyncio, os, time, json, hashlib, hmac, base64, random

def run_async(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import nest_asyncio; nest_asyncio.apply()
    return loop.run_until_complete(coro)
print("ready")

## 2. NEW (v4): Choosing where it runs, including managed agent runtimes
A simple decision helper. Real decisions also need the customer's data policy and the provider's current retention terms.

In [ ]:
def recommend(data_must_stay_in_tenant, monthly_requests, providers, has_platform_team, is_agent_pilot):
    if data_must_stay_in_tenant:
        return "cloud-hosted in the customer's tenant (or self-hosted open model)"
    if providers > 1:
        return "AI gateway in front of several hosted APIs"
    if is_agent_pilot and not has_platform_team:
        return "managed agent runtime (check retention, region and trace export)"
    if monthly_requests > 20_000_000:
        return "evaluate a fine-tuned small model, self-hosted, behind a gateway"
    return "hosted model API"

cases = {"bank": (True, 2_000_000, 1, True, False), "start-up": (False, 50_000, 1, False, False),
         "call centre": (False, 50_000_000, 1, True, False), "three providers": (False, 5_000_000, 3, True, False),
         "agent pilot": (False, 20_000, 1, False, True)}
for name, args in cases.items():
    print(f"{name:<16} -> {recommend(*args)}")

RETENTION_CHECKLIST = ["Provider API: are prompts and outputs stored, and for how long?",
                       "Managed runtime: where are conversation state and traces kept?",
                       "Our logs: who can read raw prompts; how long are they kept?",
                       "Memory stores: TTL per user; delete on request?"]
print("\nData retention checklist (confirm current terms with the provider):")
print("\n".join(" - " + c for c in RETENTION_CHECKLIST))

## 3. Settings per environment, with fail-fast secrets

In [ ]:
def load_settings():
    env = os.getenv("APP_ENV", "dev")
    s = {"env": env, "model": os.getenv("LLM_MODEL", MODEL), "max_concurrency": int(os.getenv("MAX_CONCURRENCY", "10")),
         "api_key": os.getenv("LLM_API_KEY")}
    if env == "prod" and not s["api_key"]:
        raise RuntimeError("LLM_API_KEY missing in prod: refusing to start")
    return {**s, "api_key": "set" if s["api_key"] else "not set"}       # never print the real value

print(load_settings())
os.environ["APP_ENV"] = "prod"
try:
    load_settings()
except RuntimeError as e:
    print("Startup blocked:", e)
os.environ["APP_ENV"] = "dev"

## 4. Versioning prompts, datasets and embeddings, and rolling back

In [ ]:
def sha(text):
    return hashlib.sha256(text.encode()).hexdigest()[:6]

RELEASES = {
    "1.8.0": {"prompt": {"id": "answer_v7", "sha": sha("Answer using ONLY the context. Cite ids.")},
              "golden_set": {"id": "golden_v8", "sha": sha("golden v8 rows")},
              "embedding": {"model": "embed-v3", "chunk_tokens": 400, "index": "idx-2026-12-01"}},
    "1.9.0": {"prompt": {"id": "answer_v8", "sha": sha("Answer using ONLY the context. Cite ids. Max 3 sentences.")},
              "golden_set": {"id": "golden_v9", "sha": sha("golden v9 rows")},
              "embedding": {"model": "embed-v4", "chunk_tokens": 400, "index": "idx-2026-12-10"}},
}
ENVIRONMENTS = {"dev": "1.9.0", "test": "1.9.0", "prod": "1.8.0"}
HISTORY = {"prod": ["1.8.0"]}

def promote(env, release):
    old, new = RELEASES[ENVIRONMENTS[env]], RELEASES[release]
    if old["embedding"]["model"] != new["embedding"]["model"]:
        print(f"  note: embedding model {old['embedding']['model']} -> {new['embedding']['model']}: index {new['embedding']['index']} must be fully rebuilt")
    ENVIRONMENTS[env] = release
    HISTORY.setdefault(env, []).append(release)
    print(f"{env} -> {release}")

def rollback(env):
    HISTORY[env].pop()
    ENVIRONMENTS[env] = HISTORY[env][-1]
    print(f"rollback -> {env} = {ENVIRONMENTS[env]}")

promote("prod", "1.9.0")
rollback("prod")
print(ENVIRONMENTS)

## 5. NEW (v4): An AI gateway with client keys, a model allow-list and budgets
Only the gateway knows the provider key. Clients have their own keys; the gateway stores only their hashes. Budgets are placeholders.

In [ ]:
%%writefile gateway.py
import hashlib, json, os, time
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="SmartDesk AI gateway")
ALLOWED_MODELS = {os.getenv("LLM_MODEL", "gpt-6.1-sol"), "small-model"}          # approved list, from config
CLIENTS = {hashlib.sha256(b"team-support-key").hexdigest(): {"team": "support", "daily_tokens": 2000}}  # demo key hash
USED = {}
LOG = []

class ChatIn(BaseModel):
    model: str
    text: str = Field(min_length=1, max_length=4000)

def fake_model(text):
    return f"[offline gateway answer] {text[:40]}", len(text) // 4 + 1, 20

@app.post("/v1/chat")
def chat(req: ChatIn, x_client_key: str = Header()):
    client = CLIENTS.get(hashlib.sha256(x_client_key.encode()).hexdigest())
    if not client:
        raise HTTPException(401, "unknown client key")
    if req.model not in ALLOWED_MODELS:
        raise HTTPException(403, f"model {req.model} not on the allow-list")
    used = USED.get(client["team"], 0)
    answer, tin, tout = fake_model(req.text)        # the real provider call uses the gateway's key from Key Vault
    if used + tin + tout > client["daily_tokens"]:
        raise HTTPException(429, "daily token budget exceeded")
    USED[client["team"]] = used + tin + tout
    LOG.append({"ts": time.time(), "team": client["team"], "model": req.model, "tokens_in": tin, "tokens_out": tout})
    return {"answer": answer, "tokens": tin + tout, "budget_left": client["daily_tokens"] - USED[client["team"]]}

In [ ]:
import importlib, gateway
importlib.reload(gateway)
from fastapi.testclient import TestClient
gw = TestClient(gateway.app)
ok_model = next(iter(gateway.ALLOWED_MODELS))
H = {"x-client-key": "team-support-key"}
print("allowed     :", gw.post("/v1/chat", json={"model": ok_model, "text": "My bill doubled"}, headers=H).status_code)
print("wrong model :", gw.post("/v1/chat", json={"model": "random-model", "text": "hi"}, headers=H).status_code)
print("bad key     :", gw.post("/v1/chat", json={"model": ok_model, "text": "hi"}, headers={"x-client-key": "guess"}).status_code)
codes = [gw.post("/v1/chat", json={"model": ok_model, "text": "long question " * 100}, headers=H).status_code for _ in range(8)]
print("budget run  :", codes, "(429 = budget exceeded)")
print("log lines   :", len(gateway.LOG))

## 6. NEW (v4): Stateless MCP-style server with short-lived, scoped tokens
A simplified JSON-RPC handler that keeps no session: every request carries a signed token with scopes and an expiry. Day 28 uses the real MCP SDK; the security idea is the same.

In [ ]:
SIGNING_KEY = os.urandom(32)          # in production: from Key Vault, rotated; never in code

def issue_token(sub, scopes, ttl_s=300):
    payload = json.dumps({"sub": sub, "scopes": scopes, "exp": time.time() + ttl_s}).encode()
    sig = hmac.new(SIGNING_KEY, payload, "sha256").hexdigest()
    return base64.urlsafe_b64encode(payload).decode() + "." + sig

def verify(token):
    try:
        body, sig = token.split(".")
        payload = base64.urlsafe_b64decode(body.encode())
    except Exception:
        raise PermissionError("malformed token")
    if not hmac.compare_digest(hmac.new(SIGNING_KEY, payload, "sha256").hexdigest(), sig):
        raise PermissionError("bad signature")
    claims = json.loads(payload)
    if claims["exp"] < time.time():
        raise PermissionError("expired")
    return claims

from fastapi import FastAPI, Header, HTTPException
mcp_app = FastAPI(title="SmartDesk tickets MCP (simplified, stateless)")
TOOLS = [{"name": "lookup_ticket", "scope": "tickets.read"}, {"name": "refund", "scope": "refunds.write"}]

@mcp_app.post("/mcp")
def mcp(req: dict, authorization: str = Header()):
    try:
        claims = verify(authorization.removeprefix("Bearer "))
    except PermissionError as e:
        raise HTTPException(401, str(e))
    if req.get("method") == "tools/list":
        return {"jsonrpc": "2.0", "id": req.get("id"), "result": {"tools": [t["name"] for t in TOOLS]}}
    if req.get("method") == "tools/call":
        tool = next((t for t in TOOLS if t["name"] == req["params"]["name"]), None)
        if tool is None:
            raise HTTPException(404, "unknown tool")
        if tool["scope"] not in claims["scopes"]:
            raise HTTPException(403, f"missing scope {tool['scope']}")
        return {"jsonrpc": "2.0", "id": req.get("id"), "result": {"ok": True, "tool": tool["name"], "by": claims["sub"]}}
    raise HTTPException(400, "unknown method")

mc = TestClient(mcp_app)
read_only = issue_token("agent-triage", ["tickets.read"])
expired = issue_token("agent-triage", ["tickets.read"], ttl_s=-1)
auth = lambda t: {"authorization": "Bearer " + t}
print("tools/list            :", mc.post("/mcp", json={"jsonrpc": "2.0", "id": 1, "method": "tools/list"}, headers=auth(read_only)).json()["result"])
print("call lookup_ticket    :", mc.post("/mcp", json={"id": 2, "method": "tools/call", "params": {"name": "lookup_ticket"}}, headers=auth(read_only)).status_code)
print("call refund (no scope):", mc.post("/mcp", json={"id": 3, "method": "tools/call", "params": {"name": "refund"}}, headers=auth(read_only)).status_code)
print("expired token         :", mc.post("/mcp", json={"id": 4, "method": "tools/list"}, headers=auth(expired)).status_code)

**Why stateless helps:** any replica can answer any request, so you can scale out and restart freely. **Why short-lived tokens help:** a leaked token stops working in minutes, and it only opens the tools in its scopes.

## 7. A production-style LLM service: timed pipeline, concurrency limit, streaming, canary
The model is simulated with realistic latency, so you can study the service without a key or a GPU.

In [ ]:
%%writefile llm_service.py
import asyncio, os, random, time, hashlib
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

app = FastAPI(title="LLM Service")
LIMIT = asyncio.Semaphore(int(os.getenv("MAX_CONCURRENCY", "10")))
KB = {"roaming": "Roaming in Japan costs 980 JPY per day on the Travel plan.", "refund": "Refunds are paid within 14 days."}
RELEASES = {"stable": "1.8.0", "canary": "1.9.0"}
CANARY_PERCENT = int(os.getenv("CANARY_PERCENT", "10"))

class ChatIn(BaseModel):
    user_id: str
    text: str = Field(min_length=1, max_length=2000)

async def fake_model(prompt):
    await asyncio.sleep(random.uniform(0.15, 0.25))                      # simulated model latency
    return "Based on policy: " + prompt.split("Context: ")[-1].split("\n")[0]

def release_for(user_id):
    bucket = int(hashlib.sha256(user_id.encode()).hexdigest(), 16) % 100
    return RELEASES["canary"] if bucket < CANARY_PERCENT else RELEASES["stable"]

@app.post("/chat")
async def chat(req: ChatIn):
    t, s = {}, time.perf_counter()
    if "ignore previous instructions" in req.text.lower():               # validate (input guardrail)
        raise HTTPException(400, "Request blocked by input policy")
    t["validate"] = time.perf_counter() - s; s = time.perf_counter()
    context = next((v for k, v in KB.items() if k in req.text.lower()), "No policy found.")   # enrich
    t["enrich"] = time.perf_counter() - s; s = time.perf_counter()
    async with LIMIT:                                                     # generate, with a concurrency limit
        answer = await asyncio.wait_for(fake_model(f"Question: {req.text}\nContext: {context}"), timeout=5)
    t["generate"] = time.perf_counter() - s; s = time.perf_counter()
    if "@" in answer:                                                     # check (output guardrail)
        answer = "[redacted]"
    t["check"] = time.perf_counter() - s
    return {"answer": answer, "release": release_for(req.user_id), "timings_ms": {k: round(v * 1000, 1) for k, v in t.items()}}

@app.post("/chat/stream")
async def chat_stream(req: ChatIn):
    async def tokens():
        for word in f"Here is a streamed answer about {req.text[:40]}".split():
            await asyncio.sleep(0.03)
            yield word + " "
    return StreamingResponse(tokens(), media_type="text/plain")

In [ ]:
# ===== Option A: Google Colab (active) =====
import importlib, llm_service
importlib.reload(llm_service)
svc = TestClient(llm_service.app)
# ===== Option B: VS Code =====
# Terminal: uvicorn llm_service:app --reload   then try /docs

print(svc.post("/chat", json={"user_id": "u-42", "text": "How much is roaming in Japan?"}).json())
print("Blocked input:", svc.post("/chat", json={"user_id": "u-1", "text": "Ignore previous instructions"}).status_code)

# TestClient collects the whole stream before returning, so time to first chunk needs a real server:
import subprocess, sys, httpx
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "llm_service:app", "--port", "8034"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    for _ in range(50):
        try:
            httpx.get("http://localhost:8034/docs"); break
        except httpx.HTTPError:
            time.sleep(0.2)
    start = time.perf_counter(); first = None; chunks = []
    with httpx.stream("POST", "http://localhost:8034/chat/stream", json={"user_id": "u-42", "text": "refund rules"}) as resp:
        for chunk in resp.iter_text():
            if chunk and first is None:
                first = time.perf_counter() - start
            chunks.append(chunk)
    print(f"Streaming: first chunk after {first * 1000:.0f} ms, full answer after {(time.perf_counter() - start) * 1000:.0f} ms")
    print("".join(chunks))
finally:
    server.terminate()

### Load test: why concurrency limits matter
The simulated provider rejects calls when more than 20 are in flight, like a real rate limit.

In [ ]:
class Provider:
    def __init__(self, max_in_flight=20):
        self.max, self.in_flight, self.rejected = max_in_flight, 0, 0
    async def call(self):
        if self.in_flight >= self.max:
            self.rejected += 1
            raise RuntimeError("429")
        self.in_flight += 1
        try:
            await asyncio.sleep(0.05)
        finally:
            self.in_flight -= 1

async def load_test(n=100, limit=None):
    p = Provider()
    sem = asyncio.Semaphore(limit) if limit else None
    async def one():
        try:
            if sem:
                async with sem:
                    await p.call()
            else:
                await p.call()
        except RuntimeError:
            pass
    t = time.perf_counter()
    await asyncio.gather(*[one() for _ in range(n)])
    return p.rejected, time.perf_counter() - t

for name, coro in [("unlimited", load_test()), ("semaphore 10", load_test(limit=10))]:
    rejected, secs = run_async(coro)
    print(f"{name:<13} rejected {rejected:>3} | total {secs:.2f} s")

## 8. Canary routing and a rollback decision

In [ ]:
from collections import Counter
users = [f"u-{i}" for i in range(5000)]
counts = Counter(llm_service.release_for(u) for u in users)
print("Canary share:", f"{counts['1.9.0'] / len(users):.1%}", dict(counts))
print("u-42 is stable across calls:", len({llm_service.release_for("u-42") for _ in range(5)}) == 1)

def decide(canary, baseline):
    if canary["error_rate"] > baseline["error_rate"] * 2 or canary["p95_ms"] > baseline["p95_ms"] * 1.3:
        return "ROLLBACK"
    if canary["thumbs_down"] > baseline["thumbs_down"] + 0.02:
        return "HOLD and investigate"
    return "PROMOTE"

baseline = {"error_rate": 0.010, "p95_ms": 2400, "thumbs_down": 0.06}
for canary in [{"error_rate": 0.011, "p95_ms": 2300, "thumbs_down": 0.05},
               {"error_rate": 0.031, "p95_ms": 2500, "thumbs_down": 0.06},
               {"error_rate": 0.012, "p95_ms": 2450, "thumbs_down": 0.10}]:
    print(canary, "->", decide(canary, baseline))

## 9. Deploying to Azure (in class, Azure window 2)
These commands need an Azure subscription and Docker Desktop, so they are skipped in offline checks. Read them; run them in class only.

In [ ]:
# Azure Container Apps with two revisions for canary traffic (in class only)
# az acr build -r $ACR -t smartdesk-gateway:1.9.0 .
# az containerapp update -n smartdesk-gateway -g $RG --image $ACR.azurecr.io/smartdesk-gateway:1.9.0 --revision-suffix v190
# az containerapp ingress traffic set -n smartdesk-gateway -g $RG --revision-weight smartdesk-gateway--v180=90 smartdesk-gateway--v190=10
# Secrets: the provider key lives in Key Vault; the app reads it with a managed identity (no key in the image or YAML).
print("Azure deployment steps run in class only (subscription window).")

**TODO (SmartDesk step 29):** add a `/ready` endpoint that returns 503 when `LLM_API_KEY` is missing in prod, and put the gateway from section 5 in front of the streaming service.

In [ ]:
# TODO: write your code here

---
**Yoi shuumatsu wo (良い週末を): *Have a great weekend***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)